# HCP temporal dynamics feature extraction with hctsa

RISE uses the [Highly Comparative Time-Series Analysis (hctsa)](https://github.com/benfulcher/hctsa) framework to characterize intra-regional temporal dynamics and capture individual differences in intrinsic local brain activity.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    raise FileNotFoundError("Run this notebook from the RISE repository root or pipeline directory")
from rise import (
    build_hctsa_run_inputs,
    extract_hctsa_dataset,
    find_missing_hctsa_runs,
    load_hctsa_config,
    read_subject_ids,
    validate_hctsa_environment,
)

### Configuration

`hcp_feature_parameters.jsonc` defines the time-series suffix and hctsa dependencies. The extraction is configured to compute the fixed 49 operations separately for both `NGR` and `GSR` time series for every subject and run.


In [ ]:
CONFIG_FILE = PROJECT_ROOT / "configs" / "hcp_feature_parameters.jsonc"
config = load_hctsa_config(CONFIG_FILE)
config


### Validate MATLAB and hctsa

In [ ]:
validate_hctsa_environment(config)

### Discover HCP time series

The discovery step matches existing RISE outputs by atlas and preprocessing suffix and creates one record per subject, run, and regression mode. HCP-YA contributes four resting-state runs per subject when all inputs are available.

In [ ]:
all_subjects = read_subject_ids(*config["subject_files"])

subjects = all_subjects[:2]#only a few subjects are used as examples
run_inputs = build_hctsa_run_inputs(subjects, config)
print(f"Found {len(run_inputs)} run/mode inputs for {len(subjects)} subjects.")
run_inputs[:2]

### Extract the 49 temporal-dynamics features

hctsa computes the fixed 49 operations independently for every parcel and separately for both regression modes (`NGR` and `GSR`). Parallelism is across subjects, and `skip_existing` uses mode-specific completion markers (`NGR_Done.txt` and `GSR_Done.txt`).


In [ ]:
results = extract_hctsa_dataset(
    run_inputs,
    config,
    n_jobs=config["n_jobs"],
)
results


### Check completion

Report subject/run/regression combinations that do not have a completion marker. A marker is written only after the corresponding NGR or GSR extraction finishes successfully.

In [ ]:
missing = find_missing_hctsa_runs(subjects, config)
print(f"Total missing runs: {len(missing)}")
missing
